# Exercise 1.2.19 — decompose attention scores

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `1.2 Intro to Mech Interp`  
**Notebook:** `1.2_Intro_to_Mech_Interp_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=1.2.19](https://delta-drills.vercel.app/?arena_exercise=1.2.19)


# [1.2] Intro to Mechanistic Interpretability: TransformerLens & induction circuits (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/02_[1.2]_Intro_to_Mech_Interp)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part2_intro_to_mech_interp/1.2_Intro_to_Mech_Interp_exercises.ipynb?t=20260430) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part2_intro_to_mech_interp/1.2_Intro_to_Mech_Interp_solutions.ipynb?t=20260430)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-12.png" width="350">

# Introduction

These pages are designed to get you introduced to the core concepts of mechanistic interpretability, via Neel Nanda's **TransformerLens** library.

Most of the sections are constructed in the following way:

1. A particular feature of TransformerLens is introduced.
2. You are given an exercise, in which you have to apply the feature.

The running theme of the exercises is **induction circuits**. Induction circuits are a particular type of circuit in a transformer, which can perform basic in-context learning. You should read the [corresponding section of Neel's glossary](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=_Jzi6YHRHKP1JziwdE02qdYZ), before continuing. This [LessWrong post](https://www.lesswrong.com/posts/TvrfY4c9eaGLeyDkE/induction-heads-illustrated) might also help; it contains some diagrams (like the one below) which walk through the induction mechanism step by step.

Each exercise will have a difficulty and importance rating out of 5, as well as an estimated maximum time you should spend on these exercises and sometimes a short annotation. You should interpret the ratings & time estimates relatively (e.g. if you find yourself spending about 50% longer on the exercises than the time estimates, adjust accordingly). Please do skip exercises / look at solutions if you don't feel like they're important enough to be worth doing, and you'd rather get to the good stuff!

For a lecture on the material today, which provides some high-level understanding before you dive into the material, watch the video below:

<iframe width="540" height="304" src="https://www.youtube.com/embed/lfwT79Hgytc" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram.png" width="1000">

## Content & Learning Objectives

### 1️⃣ TransformerLens: Introduction

This section is designed to get you up to speed with the TransformerLens library. You'll learn how to load and run models, and learn about the shared architecture template for all of these models (the latter of which should be familiar to you if you've already done the exercises that come before these, since many of the same design principles are followed).

> ##### Learning Objectives
>
> - Load and run a `HookedTransformer` model
> - Understand the basic architecture of these models
> - Use the model's tokenizer to convert text to tokens, and vice versa
> - Know how to cache activations, and to access activations from the cache
> - Use `circuitsvis` to visualise attention heads

### 2️⃣ Finding induction heads

Here, you'll learn about induction heads, how they work and why they are important. You'll also learn how to identify them from the characteristic induction head stripe in their attention patterns when the model input is a repeating sequence.

> ##### Learning Objectives
>
> - Understand what induction heads are, and the algorithm they are implementing
> - Inspect activation patterns to identify basic attention head patterns, and write your own functions to detect attention heads for you
> - Identify induction heads by looking at the attention patterns produced from a repeating random sequence

### 3️⃣ TransformerLens: Hooks

Next, you'll learn about hooks, which are a great feature of TransformerLens allowing you to access and intervene on activations within the model. We will mainly focus on the basics of hooks and using them to access activations (we'll mainly save the causal interventions for the later IOI exercises). You will also build some tools to perform logit attribution within your model, so you can identify which components are responsible for your model's performance on certain tasks.

> ##### Learning Objectives
>
> - Understand what hooks are, and how they are used in TransformerLens
> - Use hooks to access activations, process the results, and write them to an external tensor
> - Build tools to perform attribution, i.e. detecting which components of your model are responsible for performance on a given task
> - Understand how hooks can be used to perform basic interventions like **ablation**

### 4️⃣ Reverse-engineering induction circuits

Lastly, these exercises show you how you can reverse-engineer a circuit by looking directly at a transformer's weights (which can be considered a "gold standard" of interpretability; something not possible in every situation). You'll examine QK and OV circuits by multiplying through matrices (and learn how the FactoredMatrix class makes matrices like these much easier to analyse). You'll also look for evidence of composition between two induction heads, and once you've found it then you'll investigate the functionality of the full circuit formed from this composition.

> ##### Learning Objectives
>
> - Understand the difference between investigating a circuit by looking at activation patterns, and reverse-engineering a circuit by looking directly at the weights
> - Use the factored matrix class to inspect the QK and OV circuits within an induction circuit
> - Perform further exploration of induction circuits: composition scores, and targeted ablations

## Setup code

In [ ]:
import os
import sys
from pathlib import Path

from importlib.metadata import packages_distributions

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
if "transformer-lens" not in packages_distributions():
    %pip install transformer_lens==2.17.0 einops eindex-callum jaxtyping git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/ARENA_3.0-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import functools
import sys
from pathlib import Path
from typing import Callable

import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
from eindex import eindex
from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    utils,
)
from transformer_lens.hook_points import HookPoint

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part2_intro_to_mech_interp"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part2_intro_to_mech_interp.tests as tests
from plotly_utils import (
    hist,
    imshow,
    plot_comp_scores,
    plot_logit_attribution,
    plot_loss_difference,
)

# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)

MAIN = __name__ == "__main__"

# 1️⃣ TransformerLens: Introduction

> ##### Learning Objectives
>
> - Load and run a `HookedTransformer` model
> - Understand the basic architecture of these models
> - Use the model's tokenizer to convert text to tokens, and vice versa
> - Know how to cache activations, and to access activations from the cache
> - Use `circuitsvis` to visualise attention heads

## Introduction

*Note - most of this is written from the POV of Neel Nanda.*

This is a demo notebook for [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens), **a library I ([Neel Nanda](neelnanda.io)) wrote for doing [mechanistic interpretability](https://distill.pub/2020/circuits/zoom-in/) of GPT-2 Style language models.** The goal of mechanistic interpretability is to take a trained model and reverse engineer the algorithms the model learned during training from its weights. It is a fact about the world today that we have computer programs that can essentially speak English at a human level (GPT-3, PaLM, etc), yet we have no idea how they work nor how to write one ourselves. This offends me greatly, and I would like to solve this! Mechanistic interpretability is a very young and small field, and there are a *lot* of open problems - if you would like to help, please try working on one! **Check out my [list of concrete open problems](https://docs.google.com/document/d/1WONBzNqfKIxERejrrPlQMyKqg7jSFW92x5UMXNrMdPo/edit#) to figure out where to start.**

I wrote this library because after I left the Anthropic interpretability team and started doing independent research, I got extremely frustrated by the state of open source tooling. There's a lot of excellent infrastructure like HuggingFace and DeepSpeed to *use* or *train* models, but very little to dig into their internals and reverse engineer how they work. **This library tries to solve that**, and to make it easy to get into the field even if you don't work at an industry org with real infrastructure! The core features were heavily inspired by [Anthropic's excellent Garcon tool](https://transformer-circuits.pub/2021/garcon/index.html). Credit to Nelson Elhage and Chris Olah for building Garcon and showing me the value of good infrastructure for accelerating exploratory research!

The core design principle I've followed is to enable exploratory analysis - one of the most fun parts of mechanistic interpretability compared to normal ML is the extremely short feedback loops! The point of this library is to keep the gap between having an experiment idea and seeing the results as small as possible, to make it easy for **research to feel like play** and to enter a flow state. This notebook demonstrates how the library works and how to use it, but if you want to see how well it works for exploratory research, check out [my notebook analysing Indirect Objection Identification](https://github.com/TransformerLensOrg/TransformerLens/blob/main/demos/Exploratory_Analysis_Demo.ipynb) or [my recording of myself doing research](https://www.youtube.com/watch?v=yo4QvDn-vsU)!

## Loading and Running Models

TransformerLens comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. For this demo notebook we'll look at GPT-2 Small, an 80M parameter model, see the Available Models section for info on the rest.

In [ ]:
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

### HookedTransformerConfig

Alternatively, you can define a config object, then call `HookedTransformer.from_config(cfg)` to define your model. This is particularly useful when you want to have finer control over the architecture of your model. We'll see an example of this in the next section, when we define an attention-only model to study induction heads.

Even if you don't define your model in this way, you can still access the config object through the `cfg` attribute of the model.

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "1.2.19"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part2_intro_to_mech_interp.solutions import current_attn_detector, current_attn_detector, generate_repeated_tokens, induction_attn_detector, induction_score_hook, visualize_pattern_hook, logit_attribution, head_zero_ablation_hook, head_zero_ablation_hook, head_mean_ablation_hook, top_1_acc, decompose_qk_input, decompose_qk_input


### Exercise - decompose attention scores

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You shouldn't spend more than 5-10 minutes on this exercise.
> Having already done the previous exercises, this one should be easier.
> ```

Implement the function giving the decomposed scores (remember to scale by `sqrt(d_head)`!) For now, don't mask it.

<details>
<summary>Question - why do I focus on the attention scores, not the attention pattern? (i.e. pre softmax not post softmax)</summary>

Because the decomposition trick *only* works for things that are linear - softmax isn't linear and so we can no longer consider each component independently.
</details>

<details>
<summary>Help - I'm confused about what we're doing / why we're doing it.</summary>

Remember that each of our components writes to the residual stream separately. So after layer 0, we have:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/components.png" width="650">

We're particularly interested in the attention scores computed in head `1.4`, and how they depend on the inputs into that head. We've already decomposed the residual stream value $x$ into its terms $e$, $pe$, and $x^ 0$ through $x^{11}$ (which we've labelled $y_0, ..., y_{13}$ for simplicity), and we've done the same for key and query terms. We can picture these terms being passed into head `1.4` as:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/components-2.png" width="650">

So when we expand `attn_scores` out in full, they are a sum of $14^2 = 196$ terms - one for each combination of `(query_component, key_component)`.

---

#### Why is this decomposition useful?

We have a theory about a particular circuit in our model. We think that head `1.4` is an induction head, and the most important components that feed into this head are the prev token head `0.7` (as key) and the token embedding (as query). This is already supported by the evidence of our magnitude plots above (because we saw that `0.7` as key and token embeddings as query were large), but we still don't know how this particular key and query work **together**; we've only looked at them separately.

By decomposing `attn_scores` like this, we can check whether the contribution from combination `(query=tok_emb, key=0.7)` is indeed producing the characteristic induction head pattern which we've observed (and the other 195 terms don't really matter).
</details>

In [ ]:
def decompose_attn_scores(
    decomposed_q: Float[Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[Tensor, "k_comp k_pos d_head"],
    model: HookedTransformer,
) -> Float[Tensor, "q_comp k_comp q_pos k_pos"]:
    """
    Output is decomposed_scores with shape [query_component, key_component, query_pos, key_pos]

    The [i, j, 0, 0]th element is y_i @ W_QK @ y_j^T (so the sum along both first axes are the
    attention scores)
    """
    raise NotImplementedError()


tests.test_decompose_attn_scores(decompose_attn_scores, decomposed_q, decomposed_k, model)

<details><summary>Solution</summary>

```python
def decompose_attn_scores(
    decomposed_q: Float[Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[Tensor, "k_comp k_pos d_head"],
    model: HookedTransformer,
) -> Float[Tensor, "q_comp k_comp q_pos k_pos"]:
    """
    Output is decomposed_scores with shape [query_component, key_component, query_pos, key_pos]

    The [i, j, 0, 0]th element is y_i @ W_QK @ y_j^T (so the sum along both first axes are the
    attention scores)
    """
    return einops.einsum(
        decomposed_q,
        decomposed_k,
        "q_comp q_pos d_head, k_comp k_pos d_head -> q_comp k_comp q_pos k_pos",
    ) / (model.cfg.d_head**0.5)
```
</details>

Once these tests have passed, you can plot the results:

In [ ]:
# First plot: attention score contribution from (query_component, key_component) = (Embed, L0H7), you can replace this
# with any other pair and see that the values are generally much smaller, i.e. this pair dominates the attention score
# calculation
decomposed_scores = decompose_attn_scores(decomposed_q, decomposed_k, model)

q_label = "Embed"
k_label = "0.7"
decomposed_scores_from_pair = decomposed_scores[component_labels.index(q_label), component_labels.index(k_label)]

imshow(
    utils.to_numpy(t.tril(decomposed_scores_from_pair)),
    title=f"Attention score contributions from query = {q_label}, key = {k_label}<br>(by query & key sequence positions)",
    width=700,
)


# Second plot: std dev over query and key positions, shown by component. This shows us that the other pairs of
# (query_component, key_component) are much less important, without us having to look at each one individually like we
# did in the first plot!
decomposed_stds = einops.reduce(
    decomposed_scores, "query_decomp key_decomp query_pos key_pos -> query_decomp key_decomp", t.std
)
imshow(
    utils.to_numpy(decomposed_stds),
    labels={"x": "Key Component", "y": "Query Component"},
    title="Std dev of attn score contributions across sequence positions<br>(by query & key comp)",
    x=component_labels,
    y=component_labels,
    width=700,
)

<details>
<summary>Help - I don't understand the interpretation of these plots.</summary>

The first plot tells you that the term $e W_{QK}^{1.4} (x^{0.7})^T$ (i.e. the component of the attention scores for head `1.4` where the query is supplied by the token embeddings and the key is supplied by the output of head `0.7`) produces the distinctive attention pattern we see in the induction head: a strong diagonal stripe.

Although this tells us that this this component would probably be sufficient to implement the induction mechanism, it doesn't tell us the whole story. Ideally, we'd like to show that the other 195 terms are unimportant. Taking the standard deviation across the attention scores for a particular pair of components is a decent proxy for how important this term is in the overall attention pattern. The second plot shows us that the standard deviation is very small for all the other components, so we can be confident that the other components are unimportant.

To summarise:

* The first plot tells us that the pair `(q_component=tok_emb, k_component=0.7)` produces the characteristic induction-head pattern we see in attention head `1.4`.
* The second plot confirms that this pair is the only important one for influencing the attention pattern in `1.4`; all other pairs have very small contributions.
</details>

Note that plots like the ones above are often the most concise way of presenting a summary of the important information, and understanding what to plot is a valuable skill in any model internals-based work. However, if you want to see the "full plot" which the two plots above are both simplifications of in some sense, you can run the code below, which gives you the matrix of every single pair of components' contribution to the attention scores. So the first plot above is just a slice of the full plot below, and the second plot above is just the plot below after reducing over each slice with the standard deviation operation.

(Note - the plot you'll generate below is pretty big, so you'll want to clear it after you're done with it. If your machine is still working slowly when rendering it, you can use `fig.show(config={"staticPlot": True})` to display a non-interactive version of it.)

In [ ]:
decomposed_scores_centered = t.tril(decomposed_scores - decomposed_scores.mean(dim=-1, keepdim=True))

decomposed_scores_reshaped = einops.rearrange(
    decomposed_scores_centered,
    "q_comp k_comp q_token k_token -> (q_comp q_token) (k_comp k_token)",
)

fig = imshow(
    decomposed_scores_reshaped,
    title="Attention score contributions from all pairs of (key, query) components",
    width=1200,
    height=1200,
    return_fig=True,
)
full_seq_len = seq_len * 2 + 1
for i in range(0, full_seq_len * len(component_labels), full_seq_len):
    fig.add_hline(y=i, line_color="black", line_width=1)
    fig.add_vline(x=i, line_color="black", line_width=1)

fig.show(config={"staticPlot": True})

### Interpreting the full circuit

Now we know that head `1.4` is composing with head `0.7` via K composition, we can multiply through to create a full circuit:

$$
W_E\, W_{QK}^{1.4}\, (W_{OV}^{0.7})^T\, W_E^T
$$

and verify that it's the identity. (Note, when we say identity here, we're again thinking about it as a distribution over logits, so this should be taken to mean "high diagonal values", and we'll be using our previous metric of `top_1_acc`.)

#### Question - why should this be the identity?

<details>
<summary>Answer</summary>

This matrix is a bilinear form. Its diagonal elements $(A, A)$ are:

$$
A^T \, W_E\, W_{QK}^{1.4}\, W_{OV}^{0.7}\, W_E^T \, A = \underbrace{(A^T W_E W_Q^{1.4})}_{\text{query}} \underbrace{(A^T W_E W_{OV}^{0.7} W_K^{1.4})^T}_{\text{key}}
$$

Intuitively, the query is saying **"I'm looking for a token which followed $A$"**, and the key is saying **"I *am* a token which followed $A$"** (recall that $A^T W_E W_{OV}^{0.7}$ is the vector which gets moved one position forward by our prev token head `0.7`).

Now, consider the off-diagonal elements $(A, X)$ (for $X \neq A$). We expect these to be small, because the key doesn't match the query:

$$
A^T \, W_E\, W_{QK}^{1.4}\, W_{OV}^{0.7}\, W_E^T \, X = \underbrace{(\text{I'm looking for a token which followed A})}_\text{query} \boldsymbol{\cdot} \underbrace{(\text{I am a token which followed X})}_{\text{key}}
$$


Hence, we expect this to be the identity.

An illustration:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described-K-last.png" width="700">

<!-- ![kcomp_diagram_described-K.png](https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/kcomp_diagram_described-K.png) -->
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# Wrap tests.test_decompose_attn_scores so a passing test fires the beacon.
try:
    _dd_orig = tests.test_decompose_attn_scores
    def _dd_wrapped(*args, **kwargs):
        result = _dd_orig(*args, **kwargs)
        _dd_report_complete()
        return result
    tests.test_decompose_attn_scores = _dd_wrapped
except AttributeError:
    print('[Delta Drills] no matching test function — call _dd_report_complete() manually when done.')
